In [1]:
%load_ext sql
%sql sqlite:///master.db
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

# Задача 1.

Вывести страну, где популярнее всего группа Iron Maiden (т.е. треков куплено больше, чем в других странах). Также вывести кол-во купленных треков

In [276]:
%%sql

WITH artist_iron_maiden AS (
    SELECT
        artistId,
        name AS artist
    FROM artists
    WHERE name == "Iron Maiden"
),
albums_iron_maiden AS (
    SELECT
        albumId,
        title AS album,
        artist_iron_maiden.*
    FROM
        albums
    INNER JOIN
        artist_iron_maiden
    ON albums.artistId == artist_iron_maiden.artistId
),
tracks_iron_maiden AS (
    SELECT
        trackId,
        name AS track
    FROM
        tracks
    INNER JOIN
        albums_iron_maiden
        ON tracks.albumId == albums_iron_maiden.albumId
)

SELECT
    customers.country AS "Country",
    SUM(invoice_items.quantity) AS "Purchased Total"
FROM
    tracks_iron_maiden AS tracks_t
INNER JOIN
    invoice_items
    ON invoice_items.trackId == tracks_t.trackId
INNER JOIN
    invoices
    ON invoices.invoiceId == invoice_items.invoiceId
INNER JOIN
    customers
    ON customers.customerId == invoices.customerId
GROUP BY
    customers.country
ORDER BY
    SUM(invoice_items.quantity) DESC

LIMIT 10;

 * sqlite:///master.db
Done.


Country,Purchased Total
USA,61
Australia,28
Germany,25
Portugal,17
Canada,16
India,10
Brazil,10
Austria,10
France,9
Czech Republic,9


In [264]:
%%sql

WITH albums_iron_maiden AS (
    SELECT albums.albumId
    FROM albums
    INNER JOIN (
        SELECT artistId
        FROM artists
        WHERE name == "Iron Maiden"
    ) AS iron_maiden ON
        albums.artistId == iron_maiden.artistId
),
tracks_iron_maiden AS (
    SELECT
        tracks.trackId,
        tracks.name AS "Track"
    FROM tracks
    INNER JOIN albums_iron_maiden ON
        tracks.albumId == albums_iron_maiden.albumId
),
customers_items AS
(
    SELECT
        customers.customerId,
        customers.country,
        items.trackId,
        items.quantity
    FROM customers
    INNER JOIN invoices ON
        customers.customerId == invoices.customerId
    INNER JOIN invoice_items AS items ON
        invoices.invoiceId == items.invoiceId
)

SELECT
    csti.country,
    SUM(csti.quantity) AS sum_quantity
FROM
    tracks_iron_maiden AS trim
INNER JOIN
    customers_items AS csti
ON
    trim.trackId == csti.trackId
GROUP BY
    csti.country
ORDER BY
    sum_quantity DESC

LIMIT 10

 * sqlite:///master.db
Done.


country,sum_quantity
USA,61
Australia,28
Germany,25
Portugal,17
Canada,16
India,10
Brazil,10
Austria,10
France,9
Czech Republic,9


1. Треки группы Iron Maiden

In [277]:
%%sql

EXPLAIN SELECT
    tracks.trackId,
    tracks.name AS "Track",
    albs.title AS "Album"
FROM (
    SELECT albums.albumId, albums.title
    FROM albums
    INNER JOIN artists ON
        albums.artistId == artists.artistId
    WHERE
        artists.name == "Iron Maiden"
) AS albs
INNER JOIN tracks ON
    albs.albumId == tracks.albumId

 * sqlite:///master.db
Done.


addr,opcode,p1,p2,p3,p4,p5,comment
0,Init,0,18,0,None,0,None
1,OpenRead,3,14,0,3,0,None
2,OpenRead,1,2,0,3,0,None
3,OpenRead,2,4,0,2,0,None
4,IfEmpty,2,17,0,None,0,None
5,Rewind,3,17,0,None,0,None
6,Column,3,2,1,None,0,None
7,SeekRowid,1,16,1,None,0,None
8,Column,1,2,2,None,0,None
9,SeekRowid,2,16,2,None,0,None


In [3]:
%%sql

EXPLAIN SELECT
    tracks.trackId,
    tracks.name AS "Track",
    albs.title AS "Album"
FROM (
    SELECT albums.albumId, albums.title
    FROM albums
    INNER JOIN (
        SELECT artistId
        FROM artists
        WHERE name == "Iron Maiden"
    ) AS iron_maiden ON
        albums.artistId == iron_maiden.artistId
) AS albs
INNER JOIN tracks ON
    albs.albumId == tracks.albumId

 * sqlite:///master.db
Done.


addr,opcode,p1,p2,p3,p4,p5,comment
0,Init,0,18,0,None,0,None
1,OpenRead,4,14,0,3,0,None
2,OpenRead,1,2,0,3,0,None
3,OpenRead,3,4,0,2,0,None
4,IfEmpty,3,17,0,None,0,None
5,Rewind,4,17,0,None,0,None
6,Column,4,2,1,None,0,None
7,SeekRowid,1,16,1,None,0,None
8,Column,1,2,2,None,0,None
9,SeekRowid,3,16,2,None,0,None


Вынес подзапросы в CTE (Common Table Expression)

In [257]:
%%sql

WITH albums_iron_maiden AS (
    SELECT albums.albumId
    FROM albums
    INNER JOIN (
        SELECT artistId
        FROM artists
        WHERE name == "Iron Maiden"
    ) AS iron_maiden
        ON albums.artistId == iron_maiden.artistId
),
tracks_iron_maiden AS (
    SELECT
        tracks.trackId,
        tracks.name AS "Track"
    FROM tracks
    INNER JOIN albums_iron_maiden
        ON tracks.albumId == albums_iron_maiden.albumId
)

SELECT *
FROM tracks_iron_maiden

LIMIT 20 OFFSET 42

 * sqlite:///master.db
Done.


trackId,Track
1243,Out Of The Silent Planet
1244,The Thin Line Between Love & Hate
1245,Wildest Dreams
1246,Rainmaker
1247,No More Lies
1248,Montsegur
1249,Dance Of Death
1250,Gates Of Tomorrow
1251,New Frontier
1252,Paschendale


In [261]:
%%sql

WITH albums_iron_maiden AS (
    SELECT albums.albumId
    FROM albums
    INNER JOIN (
        SELECT artistId
        FROM artists
        WHERE name == "Iron Maiden"
    ) AS iron_maiden ON
        albums.artistId == iron_maiden.artistId
),
tracks_iron_maiden AS (
    SELECT
        tracks.trackId,
        tracks.name AS "Track"
    FROM tracks
    INNER JOIN albums_iron_maiden ON
        tracks.albumId == albums_iron_maiden.albumId
),
customers_items AS
(
    SELECT
        customers.customerId,
        customers.country,
        items.trackId,
        items.quantity
    FROM customers
    INNER JOIN invoices ON
        customers.customerId == invoices.customerId
    INNER JOIN invoice_items AS items ON
        invoices.invoiceId == items.invoiceId
)

SELECT
    csti.country,
    SUM(csti.quantity) AS sum_quantity
FROM
    tracks_iron_maiden AS trim
INNER JOIN
    customers_items AS csti
ON
    trim.trackId == csti.trackId
GROUP BY
    csti.country
ORDER BY
    sum_quantity DESC

LIMIT 10

 * sqlite:///master.db
Done.


country,sum_quantity
USA,61
Australia,28
Germany,25
Portugal,17
Canada,16
India,10
Brazil,10
Austria,10
France,9
Czech Republic,9


In [6]:
%%sql

WITH albums_iron_maiden AS
(
    SELECT albums.albumId
    FROM albums
    INNER JOIN (
        SELECT artistId
        FROM artists
        WHERE name == "Iron Maiden"
    ) AS iron_maiden
        ON albums.artistId == iron_maiden.artistId
),
tracks_iron_maiden AS
(
    SELECT
        tracks.trackId,
        tracks.name AS "Track"
    FROM tracks
    INNER JOIN albums_iron_maiden
        ON tracks.albumId == albums_iron_maiden.albumId
),
customers_items AS
(
    SELECT
        customers.customerId,
        customers.country,
        items.trackId,
        items.quantity
    FROM customers
    INNER JOIN invoices
        ON customers.customerId == invoices.customerId
    INNER JOIN invoice_items AS items
        ON invoices.invoiceId == items.invoiceId
)

SELECT *
FROM
    tracks_iron_maiden AS trim
INNER JOIN
    customers_items AS csti
    ON trim.trackId == csti.trackId
GROUP BY
    csti.country, trim.track

LIMIT 20


 * sqlite:///master.db
Done.


trackId,Track,customerId,country,trackId_1,quantity
1276,09 - Iron Maiden,55,Australia,1276,1
1404,2 A.M.,55,Australia,1404,2
1258,Afraid To Shoot Strangers,55,Australia,1258,1
1402,Blood On The World's Hands,55,Australia,1402,2
1231,Bring Your Daughter... To The Slaughter,55,Australia,1231,1
1249,Dance Of Death,55,Australia,1249,1
1303,Die With Your Boots On,55,Australia,1303,1
1240,Dream Of Mirrors,55,Australia,1240,1
1267,Fear Of The Dark,55,Australia,1267,1
1398,Fortunes Of War,55,Australia,1398,4


# Задача 2.

Выведите список треков, заказы на которые были оформлены в последний день месяца (любого) сотрудниками со стажем больше 3 лет

In [255]:
%%sql

WITH invoices_tracks AS (
    SELECT
        tracks.name AS track,
        invoices.customerId,
        invoiceDate
    FROM
        invoices
    INNER JOIN
        invoice_items
        ON invoices.invoiceId == invoice_items.invoiceId
    INNER JOIN
        tracks
        ON invoice_items.trackId == tracks.trackId
)

SELECT
    track AS "Track",
    custs.firstname || ' ' || custs.lastname AS "Name",
    strftime('%Y', datetime('now')) - strftime('%Y', hireDate) AS experience
FROM
    employees AS empls
INNER JOIN
    customers AS custs
    ON empls.employeeId == custs.supportRepId
INNER JOIN
    invoices_tracks AS invt
    ON invt.customerId == custs.customerId
WHERE
    experience > 3
    AND strftime('%d', invt.invoiceDate) == strftime('%d', date(invt.invoiceDate, 'start of month', '+1 month', '-1 day'))

LIMIT 10

 * sqlite:///master.db
Done.


Track,Name,experience
J Squared,Tim Goyer,4
Maria,Tim Goyer,4
Experiment In Terra,Luís Gonçalves,4
Take the Celestra,Luís Gonçalves,4
Pilot,François Tremblay,4
"Through the Looking Glass, Pt. 1",François Tremblay,4
Heliopolis,Terhi Hämäläinen,4
It Doesn't Matter,Terhi Hämäläinen,4
The Lost Warrior,Terhi Hämäläinen,4
"The Gun On Ice Planet Zero, Pt. 1",Terhi Hämäläinen,4


In [9]:
%%sql

PRAGMA table_info(customers)

 * sqlite:///master.db
Done.


cid,name,type,notnull,dflt_value,pk
0,CustomerId,INTEGER,1,None,1
1,FirstName,NVARCHAR (40),1,None,0
2,LastName,NVARCHAR (20),1,None,0
3,Company,NVARCHAR (80),0,None,0
4,Address,NVARCHAR (70),0,None,0
5,City,NVARCHAR (40),0,None,0
6,State,NVARCHAR (40),0,None,0
7,Country,NVARCHAR (40),0,None,0
8,PostalCode,NVARCHAR (10),0,None,0
9,Phone,NVARCHAR (24),1,None,0


In [10]:
%%sql

PRAGMA table_info(employees)

 * sqlite:///master.db
Done.


cid,name,type,notnull,dflt_value,pk
0,EmployeeId,INTEGER,1,None,1
1,LastName,NVARCHAR (20),1,None,0
2,FirstName,NVARCHAR (20),1,None,0
3,Title,NVARCHAR (30),0,None,0
4,BirthDate,DATETIME,0,None,0
5,HireDate,DATETIME,0,None,0
6,Address,NVARCHAR (70),0,None,0
7,City,NVARCHAR (40),0,None,0
8,State,NVARCHAR (40),0,None,0
9,Country,NVARCHAR (40),0,None,0


# Задача 3.

Вывести "серединный по длине" трек среди купленных в феврале 2023 года. "Серединный по длине" = медиана (не среднее!).

Медиана набора чисел — число, которое находится в середине этого набора, если его упорядочить по возрастанию, то есть такое число, что половина чисел из набора не меньше него, а другая половина не больше

In [251]:
%%sql

WITH feb23_invoices_tracks AS (
    SELECT
        tracks.name,
        tracks.milliseconds / 1000 as time_s,
        invoiceDate
    FROM
        invoices
    INNER JOIN
        invoice_items
    ON invoices.invoiceId == invoice_items.invoiceId
    INNER JOIN
        tracks
    ON invoice_items.trackId == tracks.trackId
)

SELECT
    MEDIAN(time_s) AS "Median by time_s"
FROM
    feb23_invoices_tracks
WHERE
    strftime('%Y', invoiceDate) == '2023'
    AND strftime('%m', invoiceDate) == '02'

 * sqlite:///master.db
Done.


Median by time_s
255.0


# Задача 4.

Вывести треки, встречающиеся и в нескольких плейлистах, и в нескольких заказах одновременно

In [90]:
%%sql

SELECT
    t.trackId,
    t.name,
    COUNT(pt.playlistId) AS in_playlists,
    COUNT(ii.invoiceId) AS in_invoices
FROM
    tracks AS t
INNER JOIN
    playlist_track AS pt
    ON t.trackId == pt.trackId
INNER JOIN
    invoice_items AS ii
    ON t.trackId == ii.trackId
GROUP BY
    t.trackId
HAVING
    COUNT(pt.playlistId) > 1
    AND COUNT(ii.invoiceId) > 1

LIMIT 10

 * sqlite:///master.db
Done.


TrackId,Name,in_playlists,in_invoices
2,Balls to the Wall,2,2
184,Chemical Wedding,2,2
393,Tarde Em Itapoã,2,2
858,Esquinas,2,2
867,Açai,2,2
894,Sunshine Of Your Love,4,4
1099,A Novidade (Live),2,2
1162,Perfect Crime,2,2
1168,The Garden,2,2
1169,Garden of Eden,2,2


# Задача 5.

Для каждого исполнителя, у которого есть песни нескольких жанров, найти жанр, по которому он заработал наибольшее количество денег, а также вывести процентное соотношение суммы, заработанной исполнителем по этому жанру к общей сумме заработанных денег. Формат вывода: имя исполнителя, жанр, соотношение(%)

In [265]:
%%sql

DROP VIEW IF EXISTS tracks_info;

CREATE VIEW IF NOT EXISTS tracks_info AS
    SELECT
        tracks.trackId,
        tracks.name AS track,
        artists.artistId,
        artists.name AS artist,
        albums.albumId,
        albums.title AS album,
        genres.genreId,
        genres.name AS genre,
        invoice_items.quantity * tracks.unitprice AS soldfor
    FROM
        tracks
    INNER JOIN
        genres
        ON tracks.genreId == genres.genreId
    INNER JOIN
        albums
        ON tracks.albumId == albums.albumId
    INNER JOIN
        artists
        ON albums.artistId == artists.artistId
    INNER JOIN
        invoice_items
        ON tracks.trackId == invoice_items.trackId;

SELECT *
FROM tracks_info
LIMIT 5;

 * sqlite:///master.db
Done.
Done.
Done.


TrackId,track,ArtistId,artist,AlbumId,album,GenreId,genre,soldfor
2,Balls to the Wall,2,Accept,2,Balls to the Wall,1,Rock,0.99
4,Restless and Wild,2,Accept,3,Restless and Wild,1,Rock,1.99
6,Put The Finger On You,1,AC/DC,1,For Those About To Rock We Salute You,1,Rock,6.98
8,Inject The Venom,1,AC/DC,1,For Those About To Rock We Salute You,1,Rock,7.96
10,Evil Walks,1,AC/DC,1,For Those About To Rock We Salute You,1,Rock,2.9699999999999998


In [247]:
%%sql

WITH multigenre_artists AS (
    SELECT artistId
    FROM tracks_info
    GROUP BY artistId
    HAVING COUNT(DISTINCT genreId) > 1
),
mgartists_earned_total AS (
    SELECT
        tracks_info.artistId,
        artist,
        SUM(soldfor) AS earned_total
    FROM
        tracks_info
    INNER JOIN multigenre_artists
        ON tracks_info.artistId == multigenre_artists.artistId
    GROUP BY
        artist
),
mgartists_earned_by_genre AS (
    SELECT
        tracks_info.artistId,
        artist,
        tracks_info.genreId,
        genre,
        SUM(soldfor) AS earned_by_genre
    FROM
        tracks_info
    INNER JOIN multigenre_artists
        ON tracks_info.artistId == multigenre_artists.artistId
    GROUP BY
        tracks_info.artistId, tracks_info.genreId
)

SELECT
    mgartists.artist,
    genre,
    MAX(earned_by_genre) / earned_total * 100 AS "%"
FROM
    mgartists_earned_by_genre AS mgartists
INNER JOIN
    mgartists_earned_total
    ON mgartists.artistId == mgartists_earned_total.artistId
GROUP BY
    mgartists.artist


LIMIT 30

 * sqlite:///master.db
Done.


artist,genre,%
Amy Winehouse,R&B/Soul,55.55555555555556
Antônio Carlos Jobim,Latin,62.93745346239762
Audioslave,Alternative & Punk,40.78358208955224
Battlestar Galactica,TV Shows,47.368421052631575
Eric Clapton,Blues,71.75558759913481
Faith No More,Alternative & Punk,87.0042279041675
Foo Fighters,Rock,83.36127409891031
Gilberto Gil,Latin,62.490547012856055
Guns N' Roses,Rock,65.52003706277507
Heroes,Drama,75.0
